# 01 — Why Does Python Need async?

**Goal:** Understand the problem that `asyncio` solves — and why normal threads aren't enough in Python.

## The problem

Imagine you're a chef in a restaurant. You need to:
1. Boil water (3 minutes of waiting)
2. Toast bread (2 minutes of waiting)
3. Fry an egg (1 minute of active work)

A bad chef does these sequentially: 3 + 2 + 1 = 6 minutes.
A good chef starts the water, starts the toast, then fries the egg while waiting. Total: ~3 minutes.

This is **concurrency** — handling multiple things by switching between them. You're still one person (one thread), but you're not standing idle.

Python's `asyncio` is this good chef.

## Exercise 1.1 — The sequential approach (bad chef)

In [ ]:
import time

def boil_water():
    print("Starting to boil water...")
    time.sleep(3)  # just waiting, doing nothing
    print("Water boiled!")

def toast_bread():
    print("Starting to toast bread...")
    time.sleep(2)  # just waiting, doing nothing
    print("Bread toasted!")

start = time.time()
boil_water()
toast_bread()
print(f"\nTotal time: {time.time() - start:.1f}s — we waited 5s for 0s of actual work!")

### Question 1.1
Both tasks are just WAITING (not computing). Why does Python waste 5 seconds on them?

*Your answer:*
Cai thread no chiem dung luon cpu , phai sleep() tinh day moi chay tiep !

-> 
Python chạy code tuần tự. Hàm boil_water() gọi time.sleep(3) — nó block và không return trong 3 giây. Chỉ khi nó return, code mới gọi toast_bread() và block thêm 2 giây nữa. Vì time.sleep() là blocking, ta không thể "tạm dừng" boil_water để chạy toast_bread — phải chờ từng cái một. Nên tổng là 5 giây, dù cả hai phần lớn đều chỉ chờ.

## Exercise 1.2 — The async approach (good chef)

Two new keywords:
- `async def` — this function can be paused and resumed
- `await` — "pause me here, go do something else while I wait"

In [ ]:
import asyncio

async def boil_water_async():
    print("Starting to boil water...")
    await asyncio.sleep(3)  # "go do other stuff, come back in 3s"
    print("Water boiled!")

async def toast_bread_async():
    print("Starting to toast bread...")
    await asyncio.sleep(2)  # "go do other stuff, come back in 2s"
    print("Bread toasted!")

start = time.time()
await asyncio.gather(boil_water_async(), toast_bread_async())
print(f"\nTotal time: {time.time() - start:.1f}s — both ran at the same time!")

### Question 1.2
The total time was ~3s, not 5s. But we only have ONE thread. How is this possible?

*Your answer:*
both bw go in, print "bw..." , then released the thread, then tb go in, print "...tb..." then released the thread, then tb wake up , print "Bt" , then bw wakes up , prinnt (Wb ) - right ?

## Exercise 1.3 — What `await` actually does

Let's trace the execution step by step.

In [ ]:
async def task_a():
    print("A: step 1")
    await asyncio.sleep(0)  # yield control — let others run
    print("A: step 2")
    await asyncio.sleep(0)  # yield again
    print("A: step 3")

async def task_b():
    print("B: step 1")
    await asyncio.sleep(0)
    print("B: step 2")
    await asyncio.sleep(0)
    print("B: step 3")

await asyncio.gather(task_a(), task_b())

### Question 1.3
What was the output order? Did A and B interleave? What does `await asyncio.sleep(0)` do?

**Key insight:** `await` is a voluntary pause point. Between two `await`s, your code runs uninterrupted. The event loop can ONLY switch tasks at `await` points.

*Your answer:*
A1,B1,A2,B2,A3,B3 
await -> yield control of event loop , comeback to timer -> go back to ready queue
asyncio.sleep(0) -> I dont understand, what if I just use: await sleep(0) ? We are in a corountine already ? Explain it to me !


## The critical rule

```
Between two await points, your code is the ONLY thing running.
No other task can interrupt you.
The event loop can ONLY switch when you hit an await.
```

This means:
- If you do heavy computation without `await` → everything else freezes
- If you call a blocking function (like `time.sleep`) without `await` → everything else freezes
- `await` is cooperative: YOU decide when to pause, not the system

**Next notebook:** The event loop — what's running this whole show?